# Factorized CFA — BERT sentiment + LLaMA-3-8B function classification

This notebook implements the final factorized CFA pipeline used in the study.


**Architecture**

`target-aware citation context → BERT sentiment (3 classes)`  
`target-aware citation context → LLaMA function (4 classes)`  
`predicted sentiment × predicted function → final 12-class CFA label`

The LLaMA is trained **only** on the four function labels:

- `Background_related_work`
- `Comparison`
- `Using_the_work`
- `Extending_the_work`

The final 12 labels are reconstructed deterministically after inference.

## Final experimental design

The real development pool is formed from `ACT2 + FOCAL + MultiCite` and is split once into a **70% real training set** and a **30% real validation set**. Because `Source_Paper_ID` is not available, leakage control is performed with a deterministic marker-neutral grouping key: `[TARGET_CITATION]` and `[OTHER_CITATION]` are both replaced by `[CITATION]` before normalization and hashing. Therefore, target-permutation near-duplicates from the same underlying citation context are forced into the same split.

- **Condition 1 — real training**: 70% real pool → train; 30% real pool → validation.
- **Condition 2 — synthetic augmentation**: the **same** 70% real train + `SYNTHETIC`; the **same** 30% real validation.
- `JOURNAL_REAL` is kept as the external test source in both conditions.
- Synthetic data are **training-only**. They are never placed in validation or test.

The 70/30 real split is generated with group-aware candidate search and selected to approximate balance simultaneously by source dataset, function label, and source×function strata. Exact citation duplicates with conflicting function labels are removed before splitting; same-label exact duplicates are collapsed to one row.

## Important evaluation distinction

The **direct supervised task** remains the 4-class function task:

`gold follow_up function` vs `LLaMA-predicted function`

Checkpoint selection and early stopping use the 4-class validation Macro-F1 of the citation-function classification task.

The derived 12-class evaluation is **not a fully gold 12-class evaluation**, because sentiment comes from the pretrained BERT prediction rather than from a gold sentiment annotation for these same rows.

The derived 12-class metrics are reported as downstream diagnostics by combining the fixed BERT-predicted sentiment with either the gold citation function or the LLaMA-predicted citation function.

`reference 12-class label = BERT-predicted sentiment × gold function`

`predicted 12-class label = BERT-predicted sentiment × LLaMA-predicted function`

The same BERT sentiment prediction is held fixed on both sides. Therefore, this derived metric isolates function-prediction errors conditional on the fixed BERT sentiment prediction; it is not an evaluation of BERT sentiment accuracy or a direct measurement of sentiment-error propagation.

## Post-hoc sensitivity diagnostics

After official validation/test evaluation, the notebook performs two post-hoc diagnostics:

1. **LLaMA function-confidence sensitivity**.
2. **BERT sentiment-confidence-conditioned analysis**.

Neither diagnostic changes training, checkpoint selection, official predictions, or the `JOURNAL_REAL` test labels.

In [ ]:
# ============================================================
# FINAL EXPERIMENTAL DESIGN
# ============================================================
# REAL DEVELOPMENT POOL = ACT2 + FOCAL + MultiCite
#
# CONDITION 1 — REAL TRAINING
#   TRAIN      = group-aware 70% of ACT2 + FOCAL + MultiCite
#   VALIDATION = the complementary group-aware 30% of the same REAL pool
#   TEST       = JOURNAL_REAL (external, untouched by model selection)
#
# CONDITION 2 — SYNTHETIC AUGMENTATION
#   TRAIN      = the SAME real 70% + SYNTHETIC
#   VALIDATION = the SAME real 30% (NO synthetic validation rows)
#   TEST       = the SAME JOURNAL_REAL external test
#
# Therefore the only difference between Conditions 1 and 2 is the addition
# of SYNTHETIC examples to TRAINING.
#
# Leakage protection:
#   - exact target-aware duplicates with the same function label collapse to one row
#   - exact target-aware duplicates with conflicting function labels are removed
#   - target permutations of the same context are grouped together by replacing
#     [TARGET_CITATION] and [OTHER_CITATION] with one common [CITATION] marker
#   - no marker-neutral group can cross train/validation/test
# ============================================================

In [ ]:
!pip install -qU accelerate transformers bitsandbytes peft datasets scikit-learn seaborn huggingface_hub


In [ ]:
import os
import hashlib
# Ορισμός TensorBoard Log Directory μέσω Environment Variable

# Set WANDB_API_KEY as an environment variable (never commit an API key).
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login()

In [ ]:
# Google Colab setup
from google.colab import drive
drive.mount('/content/drive')

import re
import gc
import random
import hashlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

# Interactive login for gated Meta-Llama weights.
login()

## 1. Configuration and reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = "/content/drive/MyDrive/ΔΙΔΑΚΤΟΡΙΚΟ PhD/4o PAPER"
DATA_PATH = os.path.join(PROJECT_ROOT, "unified_citation_function_4class.csv")

# Upstream BERT citation-sentiment model.
BERT_SENTIMENT_MODEL_DIR = os.path.join(
    PROJECT_ROOT,
    "pretrained_citation_polarity_recognition",
    "BERT_A1_BASE_CLINICAL_WEIGHTED_CE_2e-5",
    "saved_model",
)
BERT_SENTIMENT_TOKENIZER_DIR = os.path.join(
    PROJECT_ROOT,
    "pretrained_citation_polarity_recognition",
    "BERT_A1_BASE_CLINICAL_WEIGHTED_CE_2e-5",
    "saved_tokenizer",
)

# Factorized outputs.
FACTOR_ROOT = os.path.join(PROJECT_ROOT, "factorized_cfa")
FIG_DIR = os.path.join(FACTOR_ROOT, "fig")
REPORT_DIR = os.path.join(FACTOR_ROOT, "classification_reports")
PREDICTION_DIR = os.path.join(FACTOR_ROOT, "predictions")
SPLIT_BASE_DIR = os.path.join(FACTOR_ROOT, "splits")
TRAINING_LOG_DIR = os.path.join(FACTOR_ROOT, "logs")
MODEL_BASE_DIR = os.path.join(PROJECT_ROOT, "pretrained_models_factorized_cfa")

# Directory for the fixed group-aware real train/validation split and external test set.
SHARED_SPLIT_DIR = os.path.join(
    PROJECT_ROOT,
    "cfa_shared_splits",
    "ACT2_FOCAL_MultiCite_groupaware_70_30_seed42",
)

for directory in [
    FACTOR_ROOT, FIG_DIR, REPORT_DIR, PREDICTION_DIR,
    SPLIT_BASE_DIR, TRAINING_LOG_DIR, MODEL_BASE_DIR,
    SHARED_SPLIT_DIR,
]:
    os.makedirs(directory, exist_ok=True)

REAL_DEVELOPMENT_SOURCES = ["ACT2", "FOCAL", "MultiCite"]
TEST_SOURCE = "JOURNAL_REAL"
SYNTHETIC_SOURCE = "SYNTHETIC"
REAL_TRAIN_FRACTION = 0.70
REAL_VALIDATION_FRACTION = 0.30
SPLIT_CANDIDATES = 500
MIN_SOURCE_FUNCTION_STRATUM = 10

print("Choose experiment:")
print("1 - Train: 70% of ACT2+FOCAL+MultiCite | Validation: same real 30% | Test: JOURNAL_REAL")
print("2 - Train: same real 70% + SYNTHETIC | Validation: same real 30% | Test: JOURNAL_REAL")
DATASET_CHOICE = input("Choice (1/2): ").strip()

if DATASET_CHOICE == "1":
    USE_SYNTHETIC = False
    DATASET_TAG = "REALPOOL70_Train_REALPOOL30_Val_JournalReal_Test"
elif DATASET_CHOICE == "2":
    USE_SYNTHETIC = True
    DATASET_TAG = "REALPOOL70_SYNTHETIC_Train_REALPOOL30_Val_JournalReal_Test"
else:
    raise ValueError("Choice must be 1 or 2.")

# LLaMA-3-8B only — factorized function classifier.
LLAMA_BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
FOLDER_MODEL = "LLAMA3_8B"
FOLDER_INPUT = "input_citation"

# Main QLoRA hyperparameters intentionally preserved from the supplied notebook.
NUM_TRAIN_EPOCHS = 10
LLAMA_TRAIN_BATCH_SIZE = 16
LLAMA_GRAD_ACCUM_STEPS = 8          # effective batch = 128
LLAMA_EVAL_BATCH_SIZE = 16
LLAMA_GENERATION_BATCH_SIZE = 16
LLAMA_USE_GRADIENT_CHECKPOINTING = False
LEARNING_RATE = 2e-4
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.5
WEIGHT_DECAY = 0.01
MAX_SEQ_LEN = 256
EARLY_STOPPING_PATIENCE = 2

# Post-hoc function-confidence analysis.
FUNCTION_SCORE_TEXT_BATCH_SIZE = 2
FUNCTION_CONFIDENCE_THRESHOLDS = [0.40, 0.50, 0.60, 0.70, 0.80]

print("DATASET_TAG:", DATASET_TAG)
print("Real development sources:", REAL_DEVELOPMENT_SOURCES)
print("Synthetic used in training:", USE_SYNTHETIC)
print("Validation: fixed real 30%")
print("External test source:", TEST_SOURCE)
print("Effective train batch:", LLAMA_TRAIN_BATCH_SIZE * LLAMA_GRAD_ACCUM_STEPS)

## 2. Load data, normalize function targets, and build leakage-control keys

In [ ]:
raw_df = pd.read_csv(DATA_PATH)

required_columns = {"source_dataset", "citation", "original_label", "follow_up"}
missing_columns = required_columns - set(raw_df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

raw_df["source_dataset"] = raw_df["source_dataset"].astype(str).str.strip()

required_sources = set(REAL_DEVELOPMENT_SOURCES + [TEST_SOURCE])
if USE_SYNTHETIC:
    required_sources.add(SYNTHETIC_SOURCE)

missing_sources = required_sources - set(raw_df["source_dataset"].unique())
if missing_sources:
    raise ValueError(f"Missing required source_dataset values: {sorted(missing_sources)}")

raw_df = raw_df[raw_df["source_dataset"].isin(required_sources)].copy().reset_index(drop=True)

FUNCTION_LABELS = [
    "Background_related_work",
    "Comparison",
    "Using_the_work",
    "Extending_the_work",
]

FOLLOW_UP_ID_TO_FUNCTION = {
    0: "Background_related_work",
    1: "Comparison",
    2: "Using_the_work",
    3: "Extending_the_work",
}

def normalize_follow_up(value):
    """Accept numeric 0..3 or an already-normalized function string."""
    if pd.isna(value):
        return None
    text = str(value).strip()
    if text in FUNCTION_LABELS:
        return text
    try:
        idx = int(float(text))
    except Exception:
        return None
    return FOLLOW_UP_ID_TO_FUNCTION.get(idx)

raw_df["function_label"] = raw_df["follow_up"].apply(normalize_follow_up)
if raw_df["function_label"].isna().any():
    bad = raw_df[raw_df["function_label"].isna()][
        ["source_dataset", "citation", "follow_up"]
    ].head(20)
    raise ValueError("Invalid follow_up values found:\n" + bad.to_string(index=False))


def normalize_exact_citation(text):
    """Conservative exact-content normalization: case + whitespace only."""
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_marker_neutral_citation(text):
    """
    Remove TARGET-vs-OTHER assignment while preserving all citation positions.

    Example:
      foo [TARGET_CITATION] ... [OTHER_CITATION]
      foo [OTHER_CITATION] ... [TARGET_CITATION]

    become the same marker-neutral context and therefore the same leakage group.
    """
    text = normalize_exact_citation(text)
    text = text.replace("[target_citation]", "[citation]")
    text = text.replace("[other_citation]", "[citation]")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def stable_sha256(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


raw_df["exact_citation_key"] = raw_df["citation"].map(normalize_exact_citation)
raw_df["marker_neutral_key"] = raw_df["citation"].map(normalize_marker_neutral_citation)
raw_df["leakage_group_id"] = raw_df["marker_neutral_key"].map(stable_sha256)

# Target-marker audit.
raw_df["target_marker_count"] = raw_df["citation"].astype(str).str.count(r"\[TARGET_CITATION\]")
print("\n[TARGET_CITATION] count distribution by source:")
print(raw_df.groupby("source_dataset")["target_marker_count"].value_counts().sort_index())
non_single_target = int((raw_df["target_marker_count"] != 1).sum())
if non_single_target:
    print(
        f"WARNING: {non_single_target} rows do not contain exactly one [TARGET_CITATION]. "
        "Inspect them before treating the function label as target-specific."
    )


def clean_exact_conflicts_and_duplicates(frame, partition_name):
    """
    Cleaning rule used before splitting/evaluation:
      1) same exact target-aware citation + conflicting function labels -> remove entire conflict group
      2) same exact target-aware citation + same function label -> keep one row

    Marker-neutral groups with different TARGET positions are NOT removed; they are kept
    and later forced into the same train/validation side.
    """
    work = frame.copy().reset_index(drop=True)

    label_counts = work.groupby("exact_citation_key")["function_label"].nunique()
    conflict_keys = set(label_counts[label_counts > 1].index)
    conflict_rows = work[work["exact_citation_key"].isin(conflict_keys)].copy()

    before = len(work)
    work = work[~work["exact_citation_key"].isin(conflict_keys)].copy()
    duplicates_before_drop = int(work.duplicated(subset=["exact_citation_key"]).sum())
    work = work.drop_duplicates(subset=["exact_citation_key"], keep="first").reset_index(drop=True)

    print(f"\n=== EXACT CLEANING: {partition_name} ===")
    print("Rows before:", before)
    print("Conflicting exact groups removed:", len(conflict_keys))
    print("Rows removed due to exact label conflicts:", len(conflict_rows))
    print("Same-label exact duplicate rows collapsed:", duplicates_before_drop)
    print("Rows after:", len(work))

    return work, conflict_rows


real_pool_raw = raw_df[raw_df["source_dataset"].isin(REAL_DEVELOPMENT_SOURCES)].copy()
test_raw = raw_df[raw_df["source_dataset"].eq(TEST_SOURCE)].copy()

real_pool, real_conflicts = clean_exact_conflicts_and_duplicates(
    real_pool_raw,
    "REAL_POOL_ACT2_FOCAL_MultiCite",
)
test_df_base, test_conflicts = clean_exact_conflicts_and_duplicates(
    test_raw,
    "JOURNAL_REAL",
)

real_conflicts.to_csv(
    os.path.join(SHARED_SPLIT_DIR, "removed_exact_label_conflicts_REAL_POOL.csv"),
    index=False,
    encoding="utf-8",
)
test_conflicts.to_csv(
    os.path.join(SHARED_SPLIT_DIR, "removed_exact_label_conflicts_JOURNAL_REAL.csv"),
    index=False,
    encoding="utf-8",
)

# The external test must not share either exact or marker-neutral contexts with the real pool.
real_exact = set(real_pool["exact_citation_key"])
test_exact = set(test_df_base["exact_citation_key"])
real_groups = set(real_pool["leakage_group_id"])
test_groups = set(test_df_base["leakage_group_id"])

print("\n=== REAL POOL vs JOURNAL_REAL LEAKAGE AUDIT ===")
print("Exact normalized overlap:", len(real_exact & test_exact))
print("Marker-neutral group overlap:", len(real_groups & test_groups))

if real_exact & test_exact:
    raise ValueError("Leakage detected: exact normalized contexts overlap REAL_POOL and JOURNAL_REAL.")
if real_groups & test_groups:
    raise ValueError("Leakage detected: marker-neutral groups overlap REAL_POOL and JOURNAL_REAL.")

## 3. Create the shared group-aware 70/30 real split and build the two conditions

In [ ]:
def _candidate_balance_key(frame, validation_index, target_fraction=REAL_VALIDATION_FRACTION):
    """
    Score one candidate group split.

    Lower is better. We balance:
      - overall validation row fraction
      - source-specific validation fractions
      - function-specific validation fractions
      - source × function validation fractions (when the stratum has enough rows)

    A large coverage penalty is added if an eligible source/function or source×function
    stratum disappears completely from either train or validation.
    """
    val = frame.iloc[validation_index]
    train_mask = np.ones(len(frame), dtype=bool)
    train_mask[validation_index] = False
    train = frame.iloc[np.flatnonzero(train_mask)]

    deviations = [abs(len(val) / len(frame) - target_fraction)]
    coverage_penalty = 0

    for column in ["source_dataset", "function_label"]:
        total_counts = frame[column].value_counts()
        val_counts = val[column].value_counts()
        train_counts = train[column].value_counts()

        for key, total_n in total_counts.items():
            val_n = int(val_counts.get(key, 0))
            train_n = int(train_counts.get(key, 0))
            deviations.append(abs(val_n / total_n - target_fraction))
            if val_n == 0 or train_n == 0:
                coverage_penalty += 1

    total_cells = frame.groupby(["source_dataset", "function_label"]).size()
    val_cells = val.groupby(["source_dataset", "function_label"]).size()
    train_cells = train.groupby(["source_dataset", "function_label"]).size()

    for key, total_n in total_cells.items():
        if total_n < MIN_SOURCE_FUNCTION_STRATUM:
            continue
        val_n = int(val_cells.get(key, 0))
        train_n = int(train_cells.get(key, 0))
        deviations.append(abs(val_n / total_n - target_fraction))
        if val_n == 0 or train_n == 0:
            coverage_penalty += 1

    return (
        coverage_penalty,
        max(deviations),
        float(np.mean(deviations)),
        abs(len(val) / len(frame) - target_fraction),
    )


def create_group_aware_real_split(frame):
    splitter = GroupShuffleSplit(
        n_splits=SPLIT_CANDIDATES,
        test_size=REAL_VALIDATION_FRACTION,
        random_state=SEED,
    )

    best = None
    X_dummy = np.zeros((len(frame), 1), dtype=np.int8)
    groups = frame["leakage_group_id"].astype(str).to_numpy()

    for candidate_no, (train_index, validation_index) in enumerate(
        splitter.split(X_dummy, groups=groups),
        start=1,
    ):
        key = _candidate_balance_key(frame, validation_index)
        if best is None or key < best["key"]:
            best = {
                "key": key,
                "candidate_no": candidate_no,
                "train_index": train_index,
                "validation_index": validation_index,
            }

    if best is None:
        raise RuntimeError("No valid group-aware split candidate was produced.")

    train_real = frame.iloc[best["train_index"]].copy().reset_index(drop=True)
    validation_real = frame.iloc[best["validation_index"]].copy().reset_index(drop=True)

    print("\n=== SELECTED GROUP-AWARE REAL SPLIT ===")
    print("Candidate:", best["candidate_no"], "of", SPLIT_CANDIDATES)
    print("Balance key:", best["key"])
    print("Real train rows:", len(train_real), f"({len(train_real) / len(frame):.2%})")
    print("Real validation rows:", len(validation_real), f"({len(validation_real) / len(frame):.2%})")

    return train_real, validation_real, best


train_real_df, val_real_df, split_info = create_group_aware_real_split(real_pool)

# Hard guarantee: a marker-neutral target-permutation group can never cross train/validation.
train_real_groups = set(train_real_df["leakage_group_id"])
val_real_groups = set(val_real_df["leakage_group_id"])
if train_real_groups & val_real_groups:
    raise ValueError("Group leakage: a marker-neutral context occurs in both real train and validation.")

# Save the exact real split for reproducibility and consistent reuse across experimental conditions.
REAL_TRAIN_SHARED_PATH = os.path.join(SHARED_SPLIT_DIR, "real_train_70.csv")
REAL_VAL_SHARED_PATH = os.path.join(SHARED_SPLIT_DIR, "real_validation_30.csv")
JOURNAL_TEST_SHARED_PATH = os.path.join(SHARED_SPLIT_DIR, "journal_real_test_clean.csv")

train_real_df.to_csv(REAL_TRAIN_SHARED_PATH, index=False, encoding="utf-8")
val_real_df.to_csv(REAL_VAL_SHARED_PATH, index=False, encoding="utf-8")
test_df_base.to_csv(JOURNAL_TEST_SHARED_PATH, index=False, encoding="utf-8")

print("Shared real train:", REAL_TRAIN_SHARED_PATH)
print("Shared real validation:", REAL_VAL_SHARED_PATH)
print("Shared external test:", JOURNAL_TEST_SHARED_PATH)

# Source/function balance report for reproducibility.
def split_distribution_rows(frame, split_name):
    rows = []
    for (source, function_label), n in (
        frame.groupby(["source_dataset", "function_label"]).size().items()
    ):
        rows.append({
            "split": split_name,
            "source_dataset": source,
            "function_label": function_label,
            "n": int(n),
        })
    return rows

balance_rows = []
balance_rows += split_distribution_rows(train_real_df, "REAL_TRAIN_70")
balance_rows += split_distribution_rows(val_real_df, "REAL_VALIDATION_30")
balance_rows += split_distribution_rows(test_df_base, "JOURNAL_REAL_TEST")
balance_df = pd.DataFrame(balance_rows)
balance_df.to_csv(
    os.path.join(SHARED_SPLIT_DIR, "source_function_split_distribution.csv"),
    index=False,
)

print("\n=== REAL TRAIN SOURCE DISTRIBUTION ===")
print(train_real_df["source_dataset"].value_counts())
print("\n=== REAL VALIDATION SOURCE DISTRIBUTION ===")
print(val_real_df["source_dataset"].value_counts())
print("\n=== REAL TRAIN FUNCTION DISTRIBUTION ===")
print(train_real_df["function_label"].value_counts())
print("\n=== REAL VALIDATION FUNCTION DISTRIBUTION ===")
print(val_real_df["function_label"].value_counts())

# Build the selected condition.
train_df = train_real_df.copy()
val_df = val_real_df.copy()
test_df = test_df_base.copy()

synthetic_conflicts = pd.DataFrame(columns=raw_df.columns)
synthetic_removed_overlap = pd.DataFrame(columns=raw_df.columns)

if USE_SYNTHETIC:
    synthetic_raw = raw_df[raw_df["source_dataset"].eq(SYNTHETIC_SOURCE)].copy()
    synthetic_df, synthetic_conflicts = clean_exact_conflicts_and_duplicates(
        synthetic_raw,
        "SYNTHETIC",
    )

    # Conservative leakage control: synthetic groups that duplicate/permute ANY real
    # development context or JOURNAL_REAL context are excluded from augmentation.
    forbidden_groups = set(real_pool["leakage_group_id"]) | set(test_df["leakage_group_id"])
    overlap_mask = synthetic_df["leakage_group_id"].isin(forbidden_groups)
    synthetic_removed_overlap = synthetic_df[overlap_mask].copy()
    synthetic_df = synthetic_df[~overlap_mask].copy().reset_index(drop=True)

    print("\n=== SYNTHETIC LEAKAGE FILTER ===")
    print("Synthetic rows removed due to marker-neutral overlap with real pool/test:", len(synthetic_removed_overlap))
    print("Synthetic rows retained for TRAIN ONLY:", len(synthetic_df))

    train_df = pd.concat([train_real_df, synthetic_df], ignore_index=True)

synthetic_conflicts.to_csv(
    os.path.join(SHARED_SPLIT_DIR, "removed_exact_label_conflicts_SYNTHETIC.csv"),
    index=False,
    encoding="utf-8",
)
synthetic_removed_overlap.to_csv(
    os.path.join(SHARED_SPLIT_DIR, "removed_synthetic_overlap_with_real_or_test.csv"),
    index=False,
    encoding="utf-8",
)

# Final exact + marker-neutral leakage audit across the actual model splits.
def assert_no_pairwise_overlap(left, right, left_name, right_name):
    exact_overlap = set(left["exact_citation_key"]) & set(right["exact_citation_key"])
    group_overlap = set(left["leakage_group_id"]) & set(right["leakage_group_id"])

    print(f"{left_name} vs {right_name} exact overlap:", len(exact_overlap))
    print(f"{left_name} vs {right_name} marker-neutral overlap:", len(group_overlap))

    if exact_overlap:
        raise ValueError(f"Exact leakage detected between {left_name} and {right_name}.")
    if group_overlap:
        raise ValueError(f"Marker-neutral leakage detected between {left_name} and {right_name}.")

print("\n=== FINAL LEAKAGE AUDIT ===")
assert_no_pairwise_overlap(train_df, val_df, "TRAIN", "VALIDATION")
assert_no_pairwise_overlap(train_df, test_df, "TRAIN", "JOURNAL_REAL_TEST")
assert_no_pairwise_overlap(val_df, test_df, "VALIDATION", "JOURNAL_REAL_TEST")

print("\n=== FINAL SPLIT SIZES BEFORE BERT SENTIMENT ===")
print("Train:", len(train_df), sorted(train_df["source_dataset"].unique()))
print("Validation:", len(val_df), sorted(val_df["source_dataset"].unique()))
print("Test:", len(test_df), sorted(test_df["source_dataset"].unique()))

## 4. Obtain BERT sentiment and reconstruct the deterministic 12-class labels

In [ ]:
# Run the fixed BERT sentiment component on exactly the rows used by this condition.
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()
train_df["_split"] = "train"
val_df["_split"] = "validation"
test_df["_split"] = "test"

selected_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

sentiment_tokenizer = BertTokenizer.from_pretrained(BERT_SENTIMENT_TOKENIZER_DIR)
sentiment_model = BertForSequenceClassification.from_pretrained(BERT_SENTIMENT_MODEL_DIR)
sentiment_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sentiment_model.to(sentiment_device)
sentiment_model.eval()

texts = selected_df["citation"].astype(str).tolist()
sentiments = []
sentiment_confidences = []
prob_negative, prob_neutral, prob_positive = [], [], []
BERT_BATCH_SIZE = 16

with torch.inference_mode():
    for i in range(0, len(texts), BERT_BATCH_SIZE):
        batch_texts = texts[i:i + BERT_BATCH_SIZE]
        encoded = sentiment_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        ).to(sentiment_device)
        outputs = sentiment_model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        sentiments.extend(preds.cpu().tolist())
        sentiment_confidences.extend(probs.max(dim=1).values.cpu().tolist())
        prob_negative.extend(probs[:, 0].cpu().tolist())
        prob_neutral.extend(probs[:, 1].cpu().tolist())
        prob_positive.extend(probs[:, 2].cpu().tolist())

SENTIMENT_ID_TO_LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}
selected_df["sentiment"] = [SENTIMENT_ID_TO_LABEL.get(x, "unknown") for x in sentiments]
selected_df["sentiment_confidence"] = sentiment_confidences
selected_df["prob_negative"] = prob_negative
selected_df["prob_neutral"] = prob_neutral
selected_df["prob_positive"] = prob_positive

# BERT is not optimized jointly with LLaMA. Free its GPU memory before QLoRA training.
sentiment_model.to("cpu")
del sentiment_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

FINAL_LABEL_MAP = {
    ("Positive", "Extending_the_work"): "Constructive_extension",
    ("Positive", "Using_the_work"): "Supportive_application",
    ("Positive", "Background_related_work"): "Aligned_contextualization",
    ("Positive", "Comparison"): "Positive_comparison",
    ("Neutral", "Extending_the_work"): "Neutral_extension",
    ("Neutral", "Using_the_work"): "Neutral_application",
    ("Neutral", "Background_related_work"): "Descriptive_backgrounding",
    ("Neutral", "Comparison"): "Balanced_comparison",
    ("Negative", "Extending_the_work"): "Critical_extension",
    ("Negative", "Using_the_work"): "Misguided_application",
    ("Negative", "Background_related_work"): "Critical_backgrounding",
    ("Negative", "Comparison"): "Negative_comparison",
}

FINAL_12_LABELS = [
    "Constructive_extension",
    "Supportive_application",
    "Aligned_contextualization",
    "Positive_comparison",
    "Neutral_extension",
    "Neutral_application",
    "Descriptive_backgrounding",
    "Balanced_comparison",
    "Critical_extension",
    "Misguided_application",
    "Critical_backgrounding",
    "Negative_comparison",
]

RARE_FINAL_LABELS = {
    "Constructive_extension",
    "Supportive_application",
    "Positive_comparison",
    "Critical_extension",
    "Misguided_application",
    "Negative_comparison",
    "Neutral_extension",
    "Critical_backgrounding",
}

selected_df["reference_final_12_label"] = [
    FINAL_LABEL_MAP.get((s, f), "unknown")
    for s, f in zip(selected_df["sentiment"], selected_df["function_label"])
]
if (selected_df["reference_final_12_label"] == "unknown").any():
    raise ValueError("Some rows could not be mapped to a valid final 12-class label.")

# Factorized LLaMA input: citation context only. No sentiment/function leakage.
selected_df["input_text"] = selected_df["citation"].astype(str)

train_df = selected_df[selected_df["_split"].eq("train")].drop(columns=["_split"]).reset_index(drop=True)
val_df = selected_df[selected_df["_split"].eq("validation")].drop(columns=["_split"]).reset_index(drop=True)
test_df = selected_df[selected_df["_split"].eq("test")].drop(columns=["_split"]).reset_index(drop=True)

# Save condition-specific factorized splits after BERT annotation.
SPLIT_DIR = os.path.join(SPLIT_BASE_DIR, DATASET_TAG, FOLDER_INPUT)
os.makedirs(SPLIT_DIR, exist_ok=True)
train_df.to_csv(os.path.join(SPLIT_DIR, "train_factorized.csv"), index=False, encoding="utf-8")
val_df.to_csv(os.path.join(SPLIT_DIR, "validation_factorized.csv"), index=False, encoding="utf-8")
test_df.to_csv(os.path.join(SPLIT_DIR, "test_factorized.csv"), index=False, encoding="utf-8")

print("\n=== FINAL SPLIT SIZES AFTER BERT SENTIMENT ===")
print("Train:", len(train_df), sorted(train_df["source_dataset"].unique()))
print("Validation:", len(val_df), sorted(val_df["source_dataset"].unique()))
print("Test:", len(test_df), sorted(test_df["source_dataset"].unique()))

print("\n=== FUNCTION DISTRIBUTION BY SPLIT ===")
for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print("\n", name)
    print(part["function_label"].value_counts())

print("\n=== DERIVED REFERENCE 12-LABEL DISTRIBUTION BY SPLIT ===")
for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print("\n", name)
    print(part["reference_final_12_label"].value_counts())

## 5. Hugging Face datasets and LLaMA-3-8B QLoRA setup

In [ ]:
train_ds = Dataset.from_pandas(
    train_df[["input_text", "function_label"]].rename(
        columns={"input_text": "input", "function_label": "target"}
    ),
    preserve_index=False,
)
val_ds = Dataset.from_pandas(
    val_df[["input_text", "function_label"]].rename(
        columns={"input_text": "input", "function_label": "target"}
    ),
    preserve_index=False,
)
test_ds = Dataset.from_pandas(
    test_df[["input_text", "function_label"]].rename(
        columns={"input_text": "input", "function_label": "target"}
    ),
    preserve_index=False,
)

LLAMA_COMPUTE_DTYPE = torch.bfloat16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=LLAMA_COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(LLAMA_BASE_MODEL, use_fast=True)
tokenizer.clean_up_tokenization_spaces = False
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SPECIAL_CITATION_TOKENS = ["[TARGET_CITATION]", "[OTHER_CITATION]"]
tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_CITATION_TOKENS})
citation_marker_token_ids = tokenizer.convert_tokens_to_ids(SPECIAL_CITATION_TOKENS)

for marker, marker_id in zip(SPECIAL_CITATION_TOKENS, citation_marker_token_ids):
    marker_ids = tokenizer.encode(marker, add_special_tokens=False)
    print(marker, "->", tokenizer.tokenize(marker), "->", marker_ids)
    assert len(marker_ids) == 1 and marker_ids[0] == marker_id

model = AutoModelForCausalLM.from_pretrained(
    LLAMA_BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=LLAMA_COMPUTE_DTYPE,
    trust_remote_code=True,
)
model.resize_token_embeddings(len(tokenizer))
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=LLAMA_USE_GRADIENT_CHECKPOINTING,
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    trainable_token_indices=citation_marker_token_ids,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.generation_config.max_length = None

# Dynamically reserve enough generation tokens for the longest function label + EOT.
label_token_lengths = {
    label: len(tokenizer(label, add_special_tokens=False)["input_ids"])
    for label in FUNCTION_LABELS
}
print("Function label token lengths:", label_token_lengths)
GENERATION_MAX_NEW_TOKENS = max(label_token_lengths.values()) + 1
MAX_PROMPT_LEN = MAX_SEQ_LEN - GENERATION_MAX_NEW_TOKENS
print("GENERATION_MAX_NEW_TOKENS:", GENERATION_MAX_NEW_TOKENS)
print("MAX_PROMPT_LEN:", MAX_PROMPT_LEN)

## 6. Function prompt, safe target-aware truncation, and supervised encoding

In [ ]:
def build_llama3_messages(text):
    system_message = (
        "You are a strict citation-function classification assistant. "
        "You must answer with exactly ONE label from the list."
    )
    user_message = (
        str(text).strip()
        + "\n\nClassify the function of [TARGET_CITATION]. "
          "Treat [OTHER_CITATION] only as contextual citations."
        + "\n\nChoose ONE of the following labels:\n"
        + "\n".join(f"- {label}" for label in FUNCTION_LABELS)
        + "\n\nAnswer with ONLY the label name."
    )
    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

def llama3_eot_token_id():
    eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if eot_id is None or eot_id < 0 or eot_id == tokenizer.unk_token_id:
        raise ValueError("Tokenizer has no valid <|eot_id|> token.")
    return eot_id

def llama3_terminators():
    ids = [tokenizer.eos_token_id, llama3_eot_token_id()]
    return list(dict.fromkeys(i for i in ids if i is not None))

def centered_input_window(raw_input_ids, max_tokens):
    if len(raw_input_ids) <= max_tokens:
        return raw_input_ids
    target_id = tokenizer.convert_tokens_to_ids("[TARGET_CITATION]")
    if target_id in raw_input_ids:
        pos = raw_input_ids.index(target_id)
        start = max(0, pos - max_tokens // 2)
        start = min(start, len(raw_input_ids) - max_tokens)
    else:
        start = 0
    return raw_input_ids[start:start + max_tokens]

def _plain_ids(encoded):
    if isinstance(encoded, dict):
        ids = encoded["input_ids"]
    elif hasattr(encoded, "input_ids"):
        ids = encoded.input_ids
    else:
        ids = encoded
    if torch.is_tensor(ids):
        ids = ids.squeeze(0).tolist()
    if isinstance(ids, (list, tuple)) and ids and isinstance(ids[0], (list, tuple)):
        ids = ids[0]
    return list(ids)

def build_training_ids(input_text, target, max_len=MAX_SEQ_LEN):
    input_text = str(input_text).strip()
    target = str(target).strip()
    if target not in FUNCTION_LABELS:
        raise ValueError(f"Invalid function target: {target}")

    def encode(current_input):
        prompt_messages = build_llama3_messages(current_input)
        prompt_ids = _plain_ids(tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
        ))
        full_messages = prompt_messages + [{"role": "assistant", "content": target}]
        full_ids = _plain_ids(tokenizer.apply_chat_template(
            full_messages,
            tokenize=True,
            add_generation_prompt=False,
        ))
        if full_ids[:len(prompt_ids)] != prompt_ids:
            raise ValueError("Generation prompt is not a prefix of the full chat template.")
        assistant_ids = full_ids[len(prompt_ids):]
        if not assistant_ids or assistant_ids[-1] != llama3_eot_token_id():
            raise ValueError("Assistant target is missing or does not terminate with <|eot_id|>.")
        return prompt_ids, assistant_ids

    prompt_ids, assistant_ids = encode(input_text)

    if len(prompt_ids) + len(assistant_ids) > max_len:
        empty_prompt_ids, _ = encode("")
        available_input_tokens = max_len - len(empty_prompt_ids) - len(assistant_ids)
        if available_input_tokens <= 0:
            raise ValueError("Fixed prompt + function target does not fit in max_len.")

        raw_input_ids = tokenizer(input_text, add_special_tokens=False)["input_ids"]
        target_id = tokenizer.convert_tokens_to_ids("[TARGET_CITATION]")
        original_has_target = target_id in raw_input_ids
        budget = min(available_input_tokens, len(raw_input_ids))
        success = False

        while budget > 0:
            trimmed_ids = centered_input_window(raw_input_ids, budget)
            if original_has_target and target_id not in trimmed_ids:
                raise ValueError("Truncation removed [TARGET_CITATION].")
            trimmed_text = tokenizer.decode(trimmed_ids, skip_special_tokens=False).strip()
            prompt_ids, assistant_ids = encode(trimmed_text)
            if len(prompt_ids) + len(assistant_ids) <= max_len:
                if original_has_target:
                    check_ids = tokenizer(trimmed_text, add_special_tokens=False)["input_ids"]
                    if target_id not in check_ids:
                        budget -= 1
                        continue
                success = True
                break
            budget -= 1

        if not success:
            raise ValueError("Could not safely truncate training example.")

    input_ids = prompt_ids + assistant_ids
    labels = [-100] * len(prompt_ids) + assistant_ids
    attention_mask = [1] * len(input_ids)
    pad_len = max_len - len(input_ids)
    input_ids += [tokenizer.pad_token_id] * pad_len
    labels += [-100] * pad_len
    attention_mask += [0] * pad_len

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

def build_generation_ids(text, max_prompt_len=MAX_PROMPT_LEN):
    text = str(text).strip()

    def encode(current_text):
        return _plain_ids(tokenizer.apply_chat_template(
            build_llama3_messages(current_text),
            tokenize=True,
            add_generation_prompt=True,
        ))

    prompt_ids = encode(text)
    if len(prompt_ids) <= max_prompt_len:
        return prompt_ids

    empty_prompt_ids = encode("")
    available_input_tokens = max_prompt_len - len(empty_prompt_ids)
    if available_input_tokens <= 0:
        raise ValueError("Fixed generation prompt does not fit in max_prompt_len.")

    raw_input_ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    target_id = tokenizer.convert_tokens_to_ids("[TARGET_CITATION]")
    original_has_target = target_id in raw_input_ids
    budget = min(available_input_tokens, len(raw_input_ids))

    while budget > 0:
        trimmed_ids = centered_input_window(raw_input_ids, budget)
        if original_has_target and target_id not in trimmed_ids:
            raise ValueError("Generation truncation removed [TARGET_CITATION].")
        trimmed_text = tokenizer.decode(trimmed_ids, skip_special_tokens=False).strip()
        if original_has_target:
            check_ids = tokenizer(trimmed_text, add_special_tokens=False)["input_ids"]
            if target_id not in check_ids:
                budget -= 1
                continue
        prompt_ids = encode(trimmed_text)
        if len(prompt_ids) <= max_prompt_len:
            return prompt_ids
        budget -= 1

    raise ValueError("Could not safely truncate generation prompt.")

## 7. Generation-based validation metrics and trainer

In [ ]:
def normalize_function_label(text):
    if not isinstance(text, str) or text.strip() == "":
        return "unknown"
    text = text.strip().split("\n")[0]
    text = re.sub(r"[^\w]+$", "", text).strip()
    return text if text in FUNCTION_LABELS else "unknown"

def compute_function_metrics(gold_labels, predicted_labels):
    gold = np.asarray([normalize_function_label(x) for x in gold_labels], dtype=object)
    pred = np.asarray([normalize_function_label(x) for x in predicted_labels], dtype=object)

    invalid_gold = sorted(set(gold) - set(FUNCTION_LABELS))
    if invalid_gold:
        raise ValueError(f"Invalid gold function labels: {invalid_gold}")

    metrics = {
        "accuracy": accuracy_score(gold, pred),
        "macro_f1": f1_score(
            gold,
            pred,
            labels=FUNCTION_LABELS,
            average="macro",
            zero_division=0,
        ),
        "unknown_rate": float(np.mean(pred == "unknown")),
    }

    for label in FUNCTION_LABELS:
        key = {
            "Background_related_work": "background_f1",
            "Comparison": "comparison_f1",
            "Using_the_work": "using_f1",
            "Extending_the_work": "extending_f1",
        }[label]
        metrics[key] = f1_score(
            (gold == label).astype(int),
            (pred == label).astype(int),
            zero_division=0,
        )

    return metrics


def compute_factorized_validation_metrics(
    gold_functions,
    predicted_functions,
    sentiments,
):
    """
    Validation metrics and downstream diagnostics.

    Direct supervised metrics:
      gold function vs LLaMA-predicted function.

    Derived final-output metrics:
      reference 12-class = BERT-predicted sentiment × gold function
      predicted 12-class = BERT-predicted sentiment × predicted function

    IMPORTANT:
    The derived 12-class metric is not fully gold because BERT sentiment is
    predicted and held fixed on both sides.
    """
    metrics = compute_function_metrics(
        gold_functions,
        predicted_functions,
    )

    gold_functions = [
        normalize_function_label(x)
        for x in gold_functions
    ]
    predicted_functions = [
        normalize_function_label(x)
        for x in predicted_functions
    ]
    sentiments = [str(x).strip() for x in sentiments]

    if not (
        len(gold_functions)
        == len(predicted_functions)
        == len(sentiments)
    ):
        raise ValueError(
            "Validation arrays have different lengths: "
            f"gold={len(gold_functions)}, "
            f"pred={len(predicted_functions)}, "
            f"sentiment={len(sentiments)}"
        )

    reference_12 = [
        FINAL_LABEL_MAP.get((sentiment, function), "unknown")
        for sentiment, function in zip(sentiments, gold_functions)
    ]

    predicted_12 = [
        FINAL_LABEL_MAP.get((sentiment, function), "unknown")
        for sentiment, function in zip(sentiments, predicted_functions)
    ]

    if any(label == "unknown" for label in reference_12):
        raise ValueError(
            "Unknown label found on the reference side of "
            "derived 12-class validation."
        )

    metrics["derived12_accuracy"] = accuracy_score(
        reference_12,
        predicted_12,
    )

    metrics["derived12_macro_f1"] = f1_score(
        reference_12,
        predicted_12,
        labels=FINAL_12_LABELS,
        average="macro",
        zero_division=0,
    )

    metrics["derived12_unknown_rate"] = float(
        np.mean(
            np.asarray(predicted_12, dtype=object) == "unknown"
        )
    )

    return metrics


class LossLoggerCallback(TrainerCallback):
    def __init__(self):
        self.records = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or logs.get("epoch") is None:
            return

        wanted = {
            "epoch": logs.get("epoch"),
            "loss": logs.get("loss"),
            "eval_loss": logs.get("eval_loss"),

            # Direct 4-class function metrics.
            "eval_accuracy": logs.get("eval_accuracy"),
            "eval_macro_f1": logs.get("eval_macro_f1"),
            "eval_unknown_rate": logs.get("eval_unknown_rate"),
            "eval_background_f1": logs.get("eval_background_f1"),
            "eval_comparison_f1": logs.get("eval_comparison_f1"),
            "eval_using_f1": logs.get("eval_using_f1"),
            "eval_extending_f1": logs.get("eval_extending_f1"),

            # Derived final 12-class metrics.
            "eval_derived12_accuracy": logs.get("eval_derived12_accuracy"),
            "eval_derived12_macro_f1": logs.get("eval_derived12_macro_f1"),
            "eval_derived12_unknown_rate": logs.get("eval_derived12_unknown_rate"),
        }

        if any(v is not None for k, v in wanted.items() if k != "epoch"):
            self.records.append(wanted)


class GenerationBasedTrainer(Trainer):
    """
    Uses standard eval loss plus real autoregressive generation.

    The LLaMA is trained on the 4-class function task.
Checkpoint selection and early stopping use the 4-class validation
Macro-F1. Derived 12-class metrics are retained as downstream diagnostics.
    """
    def __init__(
        self,
        *args,
        generation_eval_texts=None,
        generation_eval_labels=None,
        generation_eval_sentiments=None,
        generation_batch_size=16,
        generation_max_new_tokens=GENERATION_MAX_NEW_TOKENS,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.generation_eval_texts = (
            list(generation_eval_texts)
            if generation_eval_texts is not None
            else None
        )
        self.generation_eval_labels = (
            list(generation_eval_labels)
            if generation_eval_labels is not None
            else None
        )
        self.generation_eval_sentiments = (
            list(generation_eval_sentiments)
            if generation_eval_sentiments is not None
            else None
        )

        self.generation_batch_size = generation_batch_size
        self.generation_max_new_tokens = generation_max_new_tokens

    def _generate_validation_labels(self):
        old_padding_side = tokenizer.padding_side
        tokenizer.padding_side = "left"
        preds = []
        self.model.eval()

        try:
            for i in range(
                0,
                len(self.generation_eval_texts),
                self.generation_batch_size,
            ):
                batch_texts = self.generation_eval_texts[
                    i:i + self.generation_batch_size
                ]

                prompt_batch = [
                    build_generation_ids(x)
                    for x in batch_texts
                ]

                features = [
                    {
                        "input_ids": ids,
                        "attention_mask": [1] * len(ids),
                    }
                    for ids in prompt_batch
                ]

                inputs = tokenizer.pad(
                    features,
                    padding=True,
                    return_tensors="pt",
                ).to(self.model.device)

                with torch.inference_mode():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=self.generation_max_new_tokens,
                        do_sample=False,
                        num_beams=1,
                        eos_token_id=llama3_terminators(),
                        pad_token_id=tokenizer.pad_token_id,
                    )

                new_ids = outputs[
                    :,
                    inputs["input_ids"].shape[1]:,
                ]

                decoded = tokenizer.batch_decode(
                    new_ids,
                    skip_special_tokens=True,
                )

                preds.extend([
                    x.strip().split("\n")[0].strip(" .,:;")
                    for x in decoded
                ])

                del inputs, outputs, new_ids

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        finally:
            tokenizer.padding_side = old_padding_side

        return preds

    def evaluation_loop(
        self,
        dataloader,
        description,
        prediction_loss_only=None,
        ignore_keys=None,
        metric_key_prefix="eval",
    ):
        output = super().evaluation_loop(
            dataloader=dataloader,
            description=description,
            prediction_loss_only=True,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )

        if (
            metric_key_prefix == "eval"
            and self.generation_eval_texts is not None
            and self.generation_eval_labels is not None
            and self.generation_eval_sentiments is not None
        ):
            generated = self._generate_validation_labels()

            metrics = compute_factorized_validation_metrics(
                gold_functions=self.generation_eval_labels,
                predicted_functions=generated,
                sentiments=self.generation_eval_sentiments,
            )

            for name, value in metrics.items():
                output.metrics[
                    f"{metric_key_prefix}_{name}"
                ] = value

        return output


## 8. Tokenize, train with early stopping on 4-class validation Macro-F1, and save


In [ ]:
def preprocess(example):
    return build_training_ids(example["input"], example["target"], max_len=MAX_SEQ_LEN)

train_ds_tok = train_ds.map(preprocess, remove_columns=train_ds.column_names)
val_ds_tok = val_ds.map(preprocess, remove_columns=val_ds.column_names)
test_ds_tok = test_ds.map(preprocess, remove_columns=test_ds.column_names)

CHECKPOINT_DIR = os.path.join(
    FACTOR_ROOT, "trainer_checkpoints", FOLDER_MODEL, FOLDER_INPUT, DATASET_TAG
)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    run_name=f"factorized_function_{FOLDER_MODEL}_{DATASET_TAG}_{FOLDER_INPUT}",
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=LLAMA_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=LLAMA_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=LLAMA_GRAD_ACCUM_STEPS,
    warmup_steps=0.05,   # 5% warmup
    weight_decay=WEIGHT_DECAY,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=False,
    bf16=True,
    optim="adamw_torch_fused",
    gradient_checkpointing=LLAMA_USE_GRADIENT_CHECKPOINTING,
    learning_rate=LEARNING_RATE,
    label_names=["labels"],
    max_grad_norm=1.0,
    report_to="none",
)

loss_logger = LossLoggerCallback()
trainer = GenerationBasedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_tok,
    eval_dataset=val_ds_tok,
    generation_eval_texts=val_df["input_text"].astype(str).tolist(),
    generation_eval_labels=val_df["function_label"].astype(str).tolist(),
    generation_eval_sentiments=val_df["sentiment"].astype(str).tolist(),
    generation_batch_size=LLAMA_GENERATION_BATCH_SIZE,
    generation_max_new_tokens=GENERATION_MAX_NEW_TOKENS,
    compute_metrics=None,
    preprocess_logits_for_metrics=None,
    callbacks=[loss_logger, EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print(
    "Checkpoint / early-stopping metric:",
    training_args.metric_for_best_model,
)
print(
    "Expected validation key:",
    "eval_macro_f1",
)

# Sanity check: supervised text should decode to a function label.
example = train_ds_tok[0]
supervised_ids = [tok for tok, lab in zip(example["input_ids"], example["labels"]) if lab != -100]
print("Decoded supervised target:", tokenizer.decode(supervised_ids))

trainer.train()

In [ ]:


# Raw Trainer log history.
logs = pd.DataFrame(trainer.state.log_history)
training_log_path = os.path.join(
    TRAINING_LOG_DIR,
    f"factorized_training_log_{FOLDER_MODEL}_{DATASET_TAG}_{FOLDER_INPUT}.csv",
)
logs.to_csv(training_log_path, index=False)
print("Training log:", training_log_path)

# Epoch-level clean table.
callback_logs = pd.DataFrame(loss_logger.records)
train_epoch_df = (
    callback_logs[callback_logs["loss"].notna()][["epoch", "loss"]]
    .rename(columns={"loss": "train_loss"})
    .drop_duplicates("epoch", keep="last")
)
eval_cols = [
    "epoch",
    "eval_loss",

    # Direct 4-class function metrics.
    "eval_accuracy",
    "eval_macro_f1",
    "eval_unknown_rate",
    "eval_background_f1",
    "eval_comparison_f1",
    "eval_using_f1",
    "eval_extending_f1",

    # Derived final 12-class metrics.
    "eval_derived12_accuracy",
    "eval_derived12_macro_f1",
    "eval_derived12_unknown_rate",
]
eval_epoch_df = (
    callback_logs[callback_logs["eval_loss"].notna()][eval_cols]
    .rename(columns={
        "eval_loss": "val_loss",
        "eval_accuracy": "val_accuracy",
        "eval_macro_f1": "val_macro_f1",
        "eval_unknown_rate": "val_unknown_rate",
        "eval_background_f1": "val_background_f1",
        "eval_comparison_f1": "val_comparison_f1",
        "eval_using_f1": "val_using_f1",
        "eval_extending_f1": "val_extending_f1",
        "eval_derived12_accuracy": "val_derived12_accuracy",
        "eval_derived12_macro_f1": "val_derived12_macro_f1",
        "eval_derived12_unknown_rate": "val_derived12_unknown_rate",
    })
    .drop_duplicates("epoch", keep="last")
)
df_loss = pd.merge(train_epoch_df, eval_epoch_df, on="epoch", how="outer").sort_values("epoch")
print(df_loss)
loss_path = os.path.join(
    TRAINING_LOG_DIR,
    f"factorized_loss_{FOLDER_MODEL}_{DATASET_TAG}_{FOLDER_INPUT}_V2.csv",
)
df_loss.to_csv(loss_path, index=False)

# Loss curve.
sns.set(style="whitegrid", context="paper", font_scale=1.0)
plt.rcParams.update({"font.size": 11})
if not df_loss.empty:
    plt.figure(figsize=(6, 4))
    plt.plot(df_loss["epoch"], df_loss["train_loss"], marker="o", label="Training Loss")
    plt.plot(df_loss["epoch"], df_loss["val_loss"], marker="o", label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIG_DIR, f"factorized_training_validation_loss_{FOLDER_MODEL}_{DATASET_TAG}_V2.pdf"),
        format="pdf", bbox_inches="tight", dpi=300,
    )
    plt.show()

# Save the trained factorized CFA adapter and tokenizer.
MODEL_SAVE_DIR = os.path.join(
    MODEL_BASE_DIR,
    FOLDER_MODEL,
    FOLDER_INPUT,
    DATASET_TAG,
)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
model.save_pretrained(os.path.join(MODEL_SAVE_DIR, "saved_model"))
tokenizer.save_pretrained(os.path.join(MODEL_SAVE_DIR, "saved_tokenizer"))
print("Factorized function model saved to:", MODEL_SAVE_DIR)

## 9. Real generation inference helper

In [ ]:
def generate_function_labels(example_texts, batch_size=LLAMA_GENERATION_BATCH_SIZE):
    model.eval()
    old_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    preds = []
    try:
        for i in range(0, len(example_texts), batch_size):
            batch = example_texts[i:i + batch_size]
            prompt_batch = [build_generation_ids(x) for x in batch]
            features = [
                {"input_ids": ids, "attention_mask": [1] * len(ids)}
                for ids in prompt_batch
            ]
            inputs = tokenizer.pad(features, padding=True, return_tensors="pt").to(model.device)
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                    do_sample=False,
                    num_beams=1,
                    eos_token_id=llama3_terminators(),
                    pad_token_id=tokenizer.pad_token_id,
                )
            new_ids = outputs[:, inputs["input_ids"].shape[1]:]
            decoded = tokenizer.batch_decode(new_ids, skip_special_tokens=True)
            preds.extend([normalize_function_label(x) for x in decoded])
            del inputs, outputs, new_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
    finally:
        tokenizer.padding_side = old_padding_side
    return preds

def final_label_from_components(sentiment, function_label):
    return FINAL_LABEL_MAP.get((sentiment, function_label), "unknown")

## 10. Validation and JOURNAL_REAL test

Primary evaluation:
- **4-class function classification**: gold `follow_up` function vs LLaMA prediction.

Derived diagnostics:
- **12-class reference comparison**: BERT-predicted sentiment × gold function vs BERT-predicted sentiment × predicted function.
- **Rare/non-rare reference comparison** derived from those same labels.

The derived 12-class and rare/non-rare metrics do **not** evaluate BERT sentiment correctness because the same BERT sentiment prediction is held fixed on both sides.

In [ ]:
def evaluate_factorized_split(split_df, split_name):
    work = split_df.copy().reset_index(drop=True)
    work["pred_function"] = generate_function_labels(work["input_text"].astype(str).tolist())
    work["gold_function"] = work["function_label"].astype(str)

    # 4-class function evaluation — this is the direct LLaMA task.
    function_metrics = compute_function_metrics(
        work["gold_function"].tolist(),
        work["pred_function"].tolist(),
    )
    print(f"\n=== {split_name}: 4-CLASS FUNCTION METRICS ===")
    print(function_metrics)
    print(classification_report(
        work["gold_function"], work["pred_function"],
        labels=FUNCTION_LABELS, zero_division=0,
    ))

    # Derived factorized 12-class labels.
    # REFERENCE side: BERT-predicted sentiment × GOLD function.
    # PREDICTION side: BERT-predicted sentiment × LLaMA-predicted function.
    # The same BERT sentiment prediction is held fixed on both sides.
    work["reference_final_12_label"] = [
        final_label_from_components(s, f)
        for s, f in zip(work["sentiment"], work["gold_function"])
    ]
    work["pred_final_12_label"] = [
        final_label_from_components(s, f)
        for s, f in zip(work["sentiment"], work["pred_function"])
    ]

    derived12_accuracy = accuracy_score(work["reference_final_12_label"], work["pred_final_12_label"])
    derived12_macro_f1 = f1_score(
        work["reference_final_12_label"], work["pred_final_12_label"],
        labels=FINAL_12_LABELS, average="macro", zero_division=0,
    )
    derived12_unknown_rate = float((work["pred_final_12_label"] == "unknown").mean())

    print(f"\n=== {split_name}: DERIVED 12-CLASS REFERENCE COMPARISON ===")
    print("Reference = BERT-predicted sentiment × gold function")
    print("Prediction = BERT-predicted sentiment × predicted function")
    print("NOTE: this is not a fully gold 12-class evaluation.")
    print("Accuracy:", derived12_accuracy)
    print("Macro-F1:", derived12_macro_f1)
    print("Unknown rate:", derived12_unknown_rate)
    print(classification_report(
        work["reference_final_12_label"], work["pred_final_12_label"],
        labels=FINAL_12_LABELS, zero_division=0,
    ))

    # Downstream rare/non-rare diagnostic under the same fixed-BERT reference.
    # This is NOT a fully gold rare/non-rare evaluation.
    work["reference_is_rare"] = work["reference_final_12_label"].isin(RARE_FINAL_LABELS).astype(int)
    work["pred_is_rare"] = work["pred_final_12_label"].isin(RARE_FINAL_LABELS).astype(int)
    derived_rare_f1 = f1_score(work["reference_is_rare"], work["pred_is_rare"], zero_division=0)
    print(f"\n=== {split_name}: DERIVED RARE / NON-RARE REFERENCE COMPARISON ===")
    print(classification_report(
        work["reference_is_rare"], work["pred_is_rare"],
        labels=[0, 1], target_names=["Non_rare", "Rare"], zero_division=0,
    ))

    # 4-class confusion matrix.
    cm = confusion_matrix(work["gold_function"], work["pred_function"], labels=FUNCTION_LABELS)
    plt.figure(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=FUNCTION_LABELS, yticklabels=FUNCTION_LABELS)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(rotation=25, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIG_DIR, f"factorized_function_confusion_{split_name}_{FOLDER_MODEL}_{DATASET_TAG}_V2.pdf"),
        format="pdf", bbox_inches="tight", dpi=300,
    )
    plt.show()

    out_path = os.path.join(
        PREDICTION_DIR,
        f"factorized_predictions_{split_name}_{FOLDER_MODEL}_{DATASET_TAG}_{FOLDER_INPUT}_V2.csv",
    )
    work.to_csv(out_path, index=False, encoding="utf-8")

    report_path = os.path.join(
        REPORT_DIR,
        f"factorized_report_{split_name}_{FOLDER_MODEL}_{DATASET_TAG}_{FOLDER_INPUT}_V2.txt",
    )
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"{split_name} — Factorized CFA\n")
        f.write("=" * 70 + "\n\n")
        f.write("4-class function report\n")
        f.write(classification_report(
            work["gold_function"], work["pred_function"],
            labels=FUNCTION_LABELS, zero_division=0,
        ))
        f.write("\n\nDerived 12-class reference report\n")
        f.write("Reference = BERT-predicted sentiment × gold function\n")
        f.write("Prediction = BERT-predicted sentiment × predicted function\n")
        f.write("This is not a fully gold 12-class evaluation.\n")
        f.write(classification_report(
            work["reference_final_12_label"], work["pred_final_12_label"],
            labels=FINAL_12_LABELS, zero_division=0,
        ))
        f.write("\n\nDerived rare/non-rare reference report\n")
        f.write(classification_report(
            work["reference_is_rare"], work["pred_is_rare"],
            labels=[0, 1], target_names=["Non_rare", "Rare"], zero_division=0,
        ))

    summary = {
        "split": split_name,
        "function_accuracy": function_metrics["accuracy"],
        "function_macro_f1": function_metrics["macro_f1"],
        "function_unknown_rate": function_metrics["unknown_rate"],
        "derived12_accuracy": derived12_accuracy,
        "derived12_macro_f1": derived12_macro_f1,
        "derived12_unknown_rate": derived12_unknown_rate,
        "derived_rare_f1": derived_rare_f1,
        "n": len(work),
    }
    return work, summary

val_results_df, val_summary = evaluate_factorized_split(val_df, "Validation_REAL_POOL_30")   #
test_results_df, test_summary = evaluate_factorized_split(test_df, "Test_JOURNAL_REAL")

summary_df = pd.DataFrame([val_summary, test_summary])
summary_path = os.path.join(
    REPORT_DIR,
    f"factorized_summary_{FOLDER_MODEL}_{DATASET_TAG}_{FOLDER_INPUT}_V2.csv",
)
summary_df.to_csv(summary_path, index=False)
print("\n=== FACTORIZED SUMMARY ===")
print(summary_df)
print("Saved:", summary_path)

## 11. LLaMA function-confidence sensitivity analysis

This is the **main confidence sensitivity analysis for the factorized function component**.

The official function prediction remains the normal generation output produced above.  
For confidence estimation only, the fine-tuned LLaMA additionally scores **all four allowed function labels** for the same citation context:

- `Background_related_work`
- `Comparison`
- `Using_the_work`
- `Extending_the_work`

For each candidate label, the notebook computes the **mean conditional log-likelihood per supervised label token** (including the terminating `<|eot_id|>` used during training). Mean rather than summed log-likelihood is used to reduce label-length bias.

The four mean log-likelihoods are then softmax-normalized **only across the four allowed candidates**. From these scores we record:

- `function_confidence_score`: normalized candidate score assigned to the **official generated function prediction**.
- `function_top_scored_label`: candidate with the largest normalized score.
- `function_top_score`: largest candidate-normalized score.
- `function_top2_score`: second-largest score.
- `function_top_margin`: `top_score - top2_score`.
- `function_generation_score_margin`: score of the official generated label minus the best competing label.
- `function_generation_scoring_agreement`: whether normal generation and candidate scoring choose the same function.

**Important:** these are model-internal comparative scores, **not calibrated probabilities of correctness**.

The sensitivity table keeps the official generated predictions fixed and recomputes:

1. 4-class function Macro-F1,
2. derived 12-class Macro-F1,
3. derived rare/non-rare F1,

for increasingly high values of `function_confidence_score`, while also reporting retention and class support.

This analysis is post-hoc only. It does **not** alter training, early stopping, checkpoint selection, the official validation/test predictions, or the final PDF rare-citation ratio.

In [ ]:
# ============================================================
# LLaMA FUNCTION-CONFIDENCE SCORING
# ============================================================

def _score_candidate_batch(input_texts, text_batch_size=FUNCTION_SCORE_TEXT_BATCH_SIZE):
    """
    Score all four FUNCTION_LABELS for every input text.

    For each candidate, we build the same training-format sequence used during
    fine-tuning and calculate mean token log-likelihood over the supervised
    assistant target tokens (function label + <|eot_id|>).

    Returns one dict per input text with:
      - candidate mean log-likelihoods
      - candidate-normalized scores
      - top label / top score / top-2 score / margin

    IMPORTANT:
    The normalized scores are comparative scores over the four candidates.
    They are NOT calibrated probabilities of correctness.
    """
    model.eval()
    output_rows = []

    for start in range(0, len(input_texts), text_batch_size):
        text_batch = [str(x) for x in input_texts[start:start + text_batch_size]]

        features = []
        metadata = []

        # Four candidate completions for every citation context.
        for local_text_index, text_value in enumerate(text_batch):
            for candidate_label in FUNCTION_LABELS:
                encoded = build_training_ids(
                    text_value,
                    candidate_label,
                    max_len=MAX_SEQ_LEN,
                )

                features.append(encoded)
                metadata.append((local_text_index, candidate_label))

        input_ids = torch.tensor(
            [x["input_ids"] for x in features],
            dtype=torch.long,
            device=model.device,
        )
        attention_mask = torch.tensor(
            [x["attention_mask"] for x in features],
            dtype=torch.long,
            device=model.device,
        )
        labels = torch.tensor(
            [x["labels"] for x in features],
            dtype=torch.long,
            device=model.device,
        )

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
            )

            # Standard causal-LM shift.
            shift_logits = outputs.logits[:, :-1, :].float()
            shift_labels = labels[:, 1:]

            supervised_mask = shift_labels.ne(-100)
            safe_targets = shift_labels.masked_fill(~supervised_mask, 0)

            target_logits = shift_logits.gather(
                dim=-1,
                index=safe_targets.unsqueeze(-1),
            ).squeeze(-1)

            # log p(target token) = target logit - logsumexp(all vocabulary logits)
            log_partition = torch.logsumexp(shift_logits, dim=-1)
            token_log_probs = target_logits - log_partition

            supervised_counts = supervised_mask.sum(dim=1).clamp_min(1)
            sequence_logprob_sum = (
                token_log_probs * supervised_mask
            ).sum(dim=1)

            # Length-normalized score to reduce bias from different label token counts.
            sequence_logprob_mean = (
                sequence_logprob_sum / supervised_counts
            )

        batch_candidate_scores = [dict() for _ in text_batch]

        for row_index, (local_text_index, candidate_label) in enumerate(metadata):
            batch_candidate_scores[local_text_index][candidate_label] = float(
                sequence_logprob_mean[row_index].detach().cpu()
            )

        # Convert four length-normalized log-scores into candidate-normalized scores.
        for score_dict in batch_candidate_scores:
            ordered_log_scores = np.asarray(
                [score_dict[label] for label in FUNCTION_LABELS],
                dtype=np.float64,
            )

            stable = ordered_log_scores - np.max(ordered_log_scores)
            exp_scores = np.exp(stable)
            normalized_scores = exp_scores / exp_scores.sum()

            candidate_score_dict = {
                label: float(score)
                for label, score in zip(FUNCTION_LABELS, normalized_scores)
            }

            ranking = sorted(
                candidate_score_dict.items(),
                key=lambda x: x[1],
                reverse=True,
            )

            top_label, top_score = ranking[0]
            second_label, second_score = ranking[1]

            output_rows.append({
                "function_candidate_mean_logprob": score_dict,
                "function_candidate_normalized_scores": candidate_score_dict,
                "function_top_scored_label": top_label,
                "function_top_score": float(top_score),
                "function_top2_scored_label": second_label,
                "function_top2_score": float(second_score),
                "function_top_margin": float(top_score - second_score),
            })

        del input_ids, attention_mask, labels, outputs
        del shift_logits, shift_labels, target_logits, log_partition, token_log_probs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return output_rows


def attach_function_confidence_scores(results_df, split_name):
    """
    Attach candidate-scoring diagnostics to the already generated official
    function predictions.

    The official prediction remains results_df['pred_function'].
    """
    work = results_df.copy().reset_index(drop=True)

    scored_rows = _score_candidate_batch(
        work["input_text"].astype(str).tolist(),
        text_batch_size=FUNCTION_SCORE_TEXT_BATCH_SIZE,
    )

    if len(scored_rows) != len(work):
        raise RuntimeError(
            f"Function-score row mismatch for {split_name}: "
            f"{len(scored_rows)} scores vs {len(work)} rows."
        )

    score_df = pd.DataFrame(scored_rows)
    work = pd.concat([work, score_df], axis=1)

    def score_for_generated_label(row):
        pred = row["pred_function"]
        scores = row["function_candidate_normalized_scores"]
        if pred not in FUNCTION_LABELS or not isinstance(scores, dict):
            return np.nan
        return float(scores[pred])

    def generated_margin(row):
        pred = row["pred_function"]
        scores = row["function_candidate_normalized_scores"]

        if pred not in FUNCTION_LABELS or not isinstance(scores, dict):
            return np.nan

        pred_score = float(scores[pred])
        competitor_scores = [
            float(score)
            for label, score in scores.items()
            if label != pred
        ]

        if not competitor_scores:
            return np.nan

        return pred_score - max(competitor_scores)

    work["function_confidence_score"] = work.apply(
        score_for_generated_label,
        axis=1,
    )

    work["function_generation_score_margin"] = work.apply(
        generated_margin,
        axis=1,
    )

    work["function_generation_scoring_agreement"] = (
        work["pred_function"].eq(work["function_top_scored_label"])
    )

    valid_conf = work["function_confidence_score"].dropna()

    print("\n" + "=" * 80)
    print(f"FUNCTION-CONFIDENCE SCORING — {split_name}")
    print("=" * 80)
    print("Rows:", len(work))
    print("Rows with valid generated-label confidence:", len(valid_conf))
    print(
        "Generation ↔ candidate-scoring agreement:",
        f"{work['function_generation_scoring_agreement'].mean():.2%}",
    )

    if not valid_conf.empty:
        print("Confidence-score mean:", float(valid_conf.mean()))
        print("Confidence-score median:", float(valid_conf.median()))
        print("Top-score margin mean:", float(work["function_top_margin"].mean()))
        print(
            "Generated-label margin mean:",
            float(work["function_generation_score_margin"].dropna().mean()),
        )

    # Save enriched predictions. This does not replace the official prediction;
    # it only adds post-hoc score columns.
    enriched_path = os.path.join(
        PREDICTION_DIR,
        f"factorized_predictions_{split_name}_{FOLDER_MODEL}_{DATASET_TAG}_"
        f"{FOLDER_INPUT}_with_function_scores_V2.csv",
    )

    # Dict columns are serialized as text by pandas CSV output.
    work.to_csv(
        enriched_path,
        index=False,
        encoding="utf-8",
    )

    print("Enriched prediction file:", enriched_path)

    return work


def function_confidence_subset_row(
    results_df,
    split_name,
    threshold=None,
):
    """
    Evaluate the OFFICIAL generated function predictions on either:
      - all rows (threshold=None), or
      - rows whose generated-label candidate-normalized score >= threshold.

    This is a post-hoc sensitivity/subset analysis only.
    """
    if threshold is None:
        subset = results_df.copy()
        subset_name = "ALL"
    else:
        conf = pd.to_numeric(
            results_df["function_confidence_score"],
            errors="coerce",
        )
        subset = results_df[conf.ge(float(threshold))].copy()
        subset_name = f">={float(threshold):.2f}"

    if subset.empty:
        return None

    function_accuracy = accuracy_score(
        subset["gold_function"],
        subset["pred_function"],
    )

    function_macro_f1 = f1_score(
        subset["gold_function"],
        subset["pred_function"],
        labels=FUNCTION_LABELS,
        average="macro",
        zero_division=0,
    )

    per_class_f1 = {}
    for function_label in FUNCTION_LABELS:
        short_name = {
            "Background_related_work": "background",
            "Comparison": "comparison",
            "Using_the_work": "using",
            "Extending_the_work": "extending",
        }[function_label]

        per_class_f1[f"{short_name}_f1"] = f1_score(
            (subset["gold_function"] == function_label).astype(int),
            (subset["pred_function"] == function_label).astype(int),
            zero_division=0,
        )

        per_class_f1[f"{short_name}_support"] = int(
            (subset["gold_function"] == function_label).sum()
        )

    derived12_macro_f1 = f1_score(
        subset["reference_final_12_label"],
        subset["pred_final_12_label"],
        labels=FINAL_12_LABELS,
        average="macro",
        zero_division=0,
    )

    derived_rare_f1 = f1_score(
        subset["reference_is_rare"],
        subset["pred_is_rare"],
        zero_division=0,
    )

    agreement_rate = float(
        subset["function_generation_scoring_agreement"].mean()
    )

    valid_margin = pd.to_numeric(
        subset["function_generation_score_margin"],
        errors="coerce",
    )

    row = {
        "split": split_name,
        "subset": subset_name,
        "function_confidence_threshold": (
            np.nan if threshold is None else float(threshold)
        ),
        "n_full": len(results_df),
        "n_retained": len(subset),
        "retention_rate": len(subset) / len(results_df),
        "function_accuracy": function_accuracy,
        "function_macro_f1": function_macro_f1,
        "derived12_macro_f1": derived12_macro_f1,
        "derived_rare_f1": derived_rare_f1,
        "generation_scoring_agreement_rate": agreement_rate,
        "mean_generated_function_margin": (
            float(valid_margin.mean())
            if valid_margin.notna().any()
            else np.nan
        ),
    }

    row.update(per_class_f1)
    return row


def run_function_confidence_sensitivity(
    results_df,
    split_name,
    thresholds=FUNCTION_CONFIDENCE_THRESHOLDS,
):
    rows = [
        function_confidence_subset_row(
            results_df,
            split_name,
            threshold=None,
        )
    ]

    for threshold in thresholds:
        row = function_confidence_subset_row(
            results_df,
            split_name,
            threshold=threshold,
        )
        if row is not None:
            rows.append(row)

    table = pd.DataFrame([row for row in rows if row is not None])

    print("\n" + "=" * 80)
    print(f"FUNCTION-CONFIDENCE SENSITIVITY — {split_name}")
    print("=" * 80)
    print(
        "Confidence = candidate-normalized sequence score assigned to "
        "the official generated function label."
    )
    print("These scores are NOT calibrated probabilities of correctness.")

    display_cols = [
        "subset",
        "n_retained",
        "retention_rate",
        "function_accuracy",
        "function_macro_f1",
        "derived12_macro_f1",
        "derived_rare_f1",
        "generation_scoring_agreement_rate",
    ]

    print(table[display_cols].to_string(index=False))

    out_path = os.path.join(
        REPORT_DIR,
        f"factorized_FUNCTION_confidence_sensitivity_"
        f"{split_name}_{DATASET_TAG}_V2.csv",
    )
    table.to_csv(out_path, index=False)

    print("Saved:", out_path)
    return table


# ------------------------------------------------------------
# Score official validation/test predictions post hoc.
# ------------------------------------------------------------
val_results_df = attach_function_confidence_scores(
    val_results_df,
    "Validation_REAL_POOL_30",
)

test_results_df = attach_function_confidence_scores(
    test_results_df,
    "Test_JOURNAL_REAL",
)

# ------------------------------------------------------------
# Run confidence-threshold sensitivity.
# ------------------------------------------------------------
val_function_confidence_sensitivity_df = run_function_confidence_sensitivity(
    val_results_df,
    "Validation_REAL_POOL_30",
)

test_function_confidence_sensitivity_df = run_function_confidence_sensitivity(
    test_results_df,
    "Test_JOURNAL_REAL",
)

# ------------------------------------------------------------
# Optional margin-based diagnostic (descriptive, no filtering applied
# to the official evaluation or to downstream inference).
# ------------------------------------------------------------
for split_name, results_df in [
    ("Validation_REAL_POOL_30", val_results_df),
    ("Test_JOURNAL_REAL", test_results_df),
]:
    margin = pd.to_numeric(
        results_df["function_generation_score_margin"],
        errors="coerce",
    )

    print("\n" + "-" * 80)
    print(f"FUNCTION MARGIN DIAGNOSTIC — {split_name}")
    print("-" * 80)
    print("Mean:", float(margin.mean()))
    print("Median:", float(margin.median()))
    print("Positive-margin rate:", float((margin > 0).mean()))

## 12. Optional BERT-confidence-conditioned subset analysis

This block is a **confidence-conditioned robustness/subset analysis**, not an analysis of BERT sentiment error propagation.

It restricts the already evaluated validation/test rows to cases where:

`BERT sentiment_confidence >= threshold`

and recomputes:

1. **4-class function Macro-F1**  
   Gold function vs LLaMA-predicted function.

2. **Derived 12-class Macro-F1**  
   Reference = BERT-predicted sentiment × gold function  
   Prediction = BERT-predicted sentiment × LLaMA-predicted function.

3. **Derived rare/non-rare F1** using the same fixed-BERT reference.

Because the **same BERT-predicted sentiment is used on both sides**, this block cannot tell us whether BERT sentiment errors caused final CFA errors. A true sentiment-error propagation analysis would require gold sentiment annotations for the same instances.

The purpose here is only to test whether the subset of rows on which BERT is more confident also exhibits different function/derived-CFA performance.

In [ ]:
def bert_confidence_conditioned_summary(results_df, split_name, threshold=0.80):
    """
    Confidence-conditioned subset analysis.

    IMPORTANT:
    - This is NOT a BERT sentiment accuracy analysis.
    - This is NOT a direct sentiment-error propagation analysis.
    - The same BERT-predicted sentiment is fixed in both the reference
      and predicted derived 12-class labels.
    """
    confidence = pd.to_numeric(
        results_df["sentiment_confidence"],
        errors="coerce",
    )

    hc = results_df[confidence.ge(float(threshold))].copy()

    if hc.empty:
        print(split_name, "— no rows at threshold", threshold)
        return None

    function_macro_f1 = f1_score(
        hc["gold_function"],
        hc["pred_function"],
        labels=FUNCTION_LABELS,
        average="macro",
        zero_division=0,
    )

    derived12_macro_f1 = f1_score(
        hc["reference_final_12_label"],
        hc["pred_final_12_label"],
        labels=FINAL_12_LABELS,
        average="macro",
        zero_division=0,
    )

    derived_rare_f1 = f1_score(
        hc["reference_is_rare"],
        hc["pred_is_rare"],
        zero_division=0,
    )

    row = {
        "split": split_name,
        "bert_sentiment_confidence_threshold": float(threshold),
        "n_full": len(results_df),
        "n_high_confidence": len(hc),
        "retention_rate": len(hc) / len(results_df),
        "function_macro_f1_high_confidence_subset": function_macro_f1,
        "derived12_macro_f1_high_confidence_subset": derived12_macro_f1,
        "derived_rare_f1_high_confidence_subset": derived_rare_f1,
    }

    print("\n" + "=" * 80)
    print(f"BERT-CONFIDENCE-CONDITIONED SUBSET — {split_name}")
    print("=" * 80)
    print(f"Threshold: sentiment_confidence >= {threshold:.2f}")
    print(f"Full N: {len(results_df)}")
    print(f"High-confidence N: {len(hc)}")
    print(f"Retention rate: {len(hc) / len(results_df):.2%}")
    print(f"4-class function Macro-F1: {function_macro_f1:.6f}")
    print(f"Derived 12-class Macro-F1: {derived12_macro_f1:.6f}")
    print(f"Derived rare/non-rare F1: {derived_rare_f1:.6f}")
    print(
        "Interpretation: these metrics describe a BERT-high-confidence subset; "
        "they do not measure BERT sentiment correctness."
    )

    return row


hc_rows = [
    bert_confidence_conditioned_summary(
        val_results_df,
        "Validation_REAL_POOL_30",
        threshold=0.80,
    ),
    bert_confidence_conditioned_summary(
        test_results_df,
        "Test_JOURNAL_REAL",
        threshold=0.80,
    ),
]

hc_rows = [x for x in hc_rows if x is not None]

if hc_rows:
    hc_df = pd.DataFrame(hc_rows)

    hc_path = os.path.join(
        REPORT_DIR,
        f"factorized_BERT_confidence_conditioned_{DATASET_TAG}_V2.csv",
    )

    hc_df.to_csv(hc_path, index=False)

    print("\n=== BERT-CONFIDENCE-CONDITIONED SUMMARY ===")
    print(hc_df)
    print("Saved:", hc_path)